[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/12.%20The%20Berth%20Allocation%20with%20Tidal%20Windows%20Problem/P12-Tier-2_executed.ipynb)

---


# 12. The Berth Allocation with Tidal Windows Problem
## Tier 2 — Hill Climbing with Random Restarts

### Goal and workflow
This notebook implements a local-search heuristic for assigning vessels to berths and choosing service start times when vessel access depends on tidal depth. It represents a candidate schedule as berth assignments and start times, evaluates its cost and feasibility, and searches for lower-cost schedules using three neighborhood moves: vessel swap, time shift, and window migration. Hill climbing accepts improving moves; random restarts repeat the search from new random feasible schedules.

The workflow follows the heuristic solution method: define the example data, define feasibility and cost, generate an initial solution, explore its neighborhood, run repeated hill climbs, and inspect the resulting schedules and performance. The heuristic does not certify global optimality.

### Inputs and outputs
The inputs are vessel drafts, processing times, arrival times and delay-cost rates; berth depth data; and tidal windows. The search uses start times on a 0.5-hour grid within a 24-hour start-time horizon. Its outputs are a feasible schedule when one is found, its objective value, search histories, and diagnostic visualizations and performance summaries.

### Model assumptions and scope
A tidal window is an access window: a vessel must have sufficient water depth when its service starts, but its processing may continue after the window closes. This represents a vessel that must enter or reach the berth during suitable tide conditions; the berth remains occupied throughout processing. The depth factor interpolates between low-tide and high-tide depth during a window. Vessels cannot overlap in time at the same berth. Vessel length, spatial separation along a berth, weather, uncertainty in arrival or processing times, and constraints beyond the 24-hour start-time horizon are not modeled. The berth length is retained as input data but does not constrain assignments because vessel lengths are not part of this example.

### Example data
The example has three vessels. Vessel A has a draft of 12 m, a processing time of 4 h, and an arrival time of 06:00; Vessel B has a draft of 14 m, a processing time of 6 h, and an arrival time of 08:00; Vessel C has a draft of 11 m, a processing time of 3 h, and an arrival time of 14:00. The two high-water access windows are 06:30–09:30 and 19:00–22:00. Times in the code are measured in hours from the beginning of the planning day.

### What to examine
Observe how feasibility depends on both draft and access time, how each neighborhood operator changes a candidate schedule, and how restart count affects the best schedule found in the experiment. Interpret the result as the best feasible schedule found by this finite stochastic search—not as a proof that no better schedule exists.

## 1. Mathematical model and search representation

A schedule makes two decisions for each vessel: which berth it uses and when service starts. Let $V$ be the set of vessels, $B$ the set of berths, and $W$ the set of tidal windows. For vessel $i\in V$, let $a_i$ be its arrival time, $p_i$ its processing time, $d_i$ its required draft, and $c_i$ its delay-cost rate. For berth $b\in B$, let $D_b^L$ and $D_b^H$ be its low- and high-tide depths. A candidate solution uses berth assignment $b_i\in B$ and start time $s_i$ for each vessel.

The objective is the total cost of elapsed time from arrival through completion:

$$
\min \quad \sum_{i\in V} c_i\bigl(s_i+p_i-a_i\bigr).
$$

Here $s_i+p_i$ is the completion time, so $s_i+p_i-a_i$ includes both waiting after arrival and the vessel's processing time. The code evaluates this expression in `calculate_objective` and stores the result in `Solution.objective_value`.

A vessel can start only after it arrives and during a tidal access window that provides enough depth at its assigned berth. If $f(t)$ is the depth factor at time $t$, the available depth at berth $b$ is $D_b^L+f(t)(D_b^H-D_b^L)$, where $f(t)$ equals the window's `depth_factor` during a window and is zero otherwise. The draft condition is

$$
s_i\geq a_i, \qquad d_i\leq D_{b_i}^L+f(s_i)\bigl(D_{b_i}^H-D_{b_i}^L\bigr),
$$

and $s_i$ must lie within a tidal window. The entire service interval need not fit inside that window under the access-window assumption described above. For any two distinct vessels $i$ and $j$ assigned to the same berth, their occupied service intervals must not overlap:

$$
b_i=b_j \quad\Longrightarrow\quad s_i+p_i\leq s_j\ \text{or}\ s_j+p_j\leq s_i.
$$

The search discretizes possible start times in 0.5-hour increments. A state is one complete assignment of berths and start times; a neighborhood move creates a related candidate state. A move is accepted only when the resulting schedule is feasible and has a strictly lower objective value. Hill climbing stops when none of the sampled operator moves improves the current state or when the iteration limit is reached. A random restart begins another hill climb from a newly generated feasible schedule and preserves the best solution found across restarts.

The method explores only sampled neighbors on this time grid, so it can stop at a local optimum: a state with no improving neighbor among the moves considered, even though a better state may exist elsewhere. Restarts explore different starting states but do not provide an optimality guarantee. With $R$ restarts, $I$ iterations per restart, and $N=|V|$ vessels, checking a candidate schedule requires sorting service intervals by berth and takes $O(N\log N)$ in the worst case. The hill-climbing search therefore takes at most $O(RI N\log N)$ for neighborhood evaluation, in addition to the cost of generating initial schedules.

In [ ]:
# Import the numerical, plotting, and standard-library tools used in this heuristic.
import copy
import math
import random
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np

# Fix both random-number generators so the search is reproducible when rerun.
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully.")

In [ ]:
@dataclass
class Vessel:
    """Operational characteristics and delay-cost rate for a vessel."""

    id: int
    draft: float  # Required water depth in meters
    processing_time: float  # Service duration in hours
    arrival_time: float  # Earliest arrival time in hours from the planning-day start
    demurrage_cost: float  # Cost per hour from arrival through completion


@dataclass
class Berth:
    """Depth and length characteristics for a berth."""

    id: int
    length: float  # Berth length in meters; not a constraint in this example
    depth_low_tide: float  # Available depth outside a tidal window, in meters
    depth_high_tide: float  # Available depth at a window's full depth factor, in meters


@dataclass
class TidalWindow:
    """Time interval during which a tidal depth factor is available."""

    start_time: float  # Window start time in hours
    end_time: float  # Window end time in hours
    depth_factor: float  # Fraction of the low-to-high depth increase, from 0 to 1


@dataclass
class Solution:
    """Complete berth assignment and start-time schedule with its evaluation."""

    vessel_assignments: Dict[int, int]  # Vessel ID to berth ID
    start_times: Dict[int, float]  # Vessel ID to service start time
    objective_value: float  # Total weighted elapsed time cost
    feasible: bool  # Whether all modeled constraints are satisfied

In [ ]:
# Define the concrete three-vessel example.
vessels = [
    Vessel(id=0, draft=12.0, processing_time=4.0, arrival_time=6.0, demurrage_cost=8000),
    Vessel(id=1, draft=14.0, processing_time=6.0, arrival_time=8.0, demurrage_cost=12000),
    Vessel(id=2, draft=11.0, processing_time=3.0, arrival_time=14.0, demurrage_cost=6000),
]

berths = [
    Berth(id=0, length=300.0, depth_low_tide=10.0, depth_high_tide=15.0),
    Berth(id=1, length=350.0, depth_low_tide=11.0, depth_high_tide=16.0),
]

# The windows specify periods of elevated access depth within a 24-hour day.
tidal_windows = [
    TidalWindow(start_time=6.5, end_time=9.5, depth_factor=1.0),
    TidalWindow(start_time=19.0, end_time=22.0, depth_factor=1.0),
]

assert len({vessel.id for vessel in vessels}) == len(vessels)
assert len({berth.id for berth in berths}) == len(berths)
assert all(0.0 <= window.depth_factor <= 1.0 for window in tidal_windows)

print(
    f"Problem setup: {len(vessels)} vessels, {len(berths)} berths, "
    f"{len(tidal_windows)} tidal windows."
)
print(
    "Vessels (ID, draft in meters, processing time in hours, arrival time in hours): "
    f"{[(v.id, v.draft, v.processing_time, v.arrival_time) for v in vessels]}"
)
print(
    "Berths (ID, low-tide depth in meters, high-tide depth in meters): "
    f"{[(b.id, b.depth_low_tide, b.depth_high_tide) for b in berths]}"
)

In [ ]:
def get_tidal_depth(time: float, berth: Berth) -> float:
    """Return the berth's available depth at a time, including window depth factors."""
    for window in tidal_windows:
        if window.start_time <= time <= window.end_time:
            depth_increase = berth.depth_high_tide - berth.depth_low_tide
            return berth.depth_low_tide + window.depth_factor * depth_increase

    return berth.depth_low_tide


def is_feasible_assignment(vessel: Vessel, berth: Berth, start_time: float) -> bool:
    """Check arrival, tidal-access, and draft requirements for one vessel assignment."""
    if start_time < vessel.arrival_time:
        return False

    # The tidal window governs access at the service start; processing may continue after it.
    in_access_window = any(
        window.start_time <= start_time <= window.end_time
        for window in tidal_windows
    )
    if not in_access_window:
        return False

    return get_tidal_depth(start_time, berth) >= vessel.draft


def calculate_objective(solution: Solution) -> float:
    """Calculate total delay-cost rate times elapsed time from arrival to completion."""
    total_cost = 0.0

    for vessel_id in solution.vessel_assignments:
        vessel = vessels[vessel_id]
        start_time = solution.start_times[vessel_id]
        completion_time = start_time + vessel.processing_time
        elapsed_time = completion_time - vessel.arrival_time
        total_cost += vessel.demurrage_cost * elapsed_time

    return total_cost


def is_solution_feasible(solution: Solution) -> bool:
    """Check complete assignments, individual constraints, and berth non-overlap."""
    vessel_ids = {vessel.id for vessel in vessels}
    berth_ids = {berth.id for berth in berths}

    if set(solution.vessel_assignments) != vessel_ids:
        return False
    if set(solution.start_times) != vessel_ids:
        return False

    berth_by_id = {berth.id: berth for berth in berths}
    vessel_by_id = {vessel.id: vessel for vessel in vessels}
    berth_schedules = {berth_id: [] for berth_id in berth_ids}

    for vessel_id, berth_id in solution.vessel_assignments.items():
        if berth_id not in berth_by_id:
            return False

        vessel = vessel_by_id[vessel_id]
        berth = berth_by_id[berth_id]
        start_time = solution.start_times[vessel_id]
        if not is_feasible_assignment(vessel, berth, start_time):
            return False

        end_time = start_time + vessel.processing_time
        berth_schedules[berth_id].append((start_time, end_time))

    # Sorting makes it sufficient to compare consecutive service intervals at each berth.
    for schedule in berth_schedules.values():
        schedule.sort()
        for index in range(len(schedule) - 1):
            if schedule[index][1] > schedule[index + 1][0]:
                return False

    return True


print("Feasibility and objective functions defined.")

In [ ]:
def generate_random_solution() -> Solution:
    """Sample random berth assignments and start times until a feasible schedule is found."""
    max_attempts = 1000

    for _ in range(max_attempts):
        vessel_assignments = {
            vessel.id: random.choice(berths).id
            for vessel in vessels
        }
        start_times = {}
        assignments_have_start_times = True

        for vessel in vessels:
            berth_id = vessel_assignments[vessel.id]
            berth = next(berth for berth in berths if berth.id == berth_id)
            feasible_times = [
                float(candidate_time)
                for candidate_time in np.arange(vessel.arrival_time, 24.0, 0.5)
                if is_feasible_assignment(vessel, berth, float(candidate_time))
            ]

            if not feasible_times:
                assignments_have_start_times = False
                break

            start_times[vessel.id] = random.choice(feasible_times)

        if not assignments_have_start_times:
            continue

        solution = Solution(
            vessel_assignments=vessel_assignments,
            start_times=start_times,
            objective_value=0.0,
            feasible=False,
        )
        solution.feasible = is_solution_feasible(solution)
        if solution.feasible:
            solution.objective_value = calculate_objective(solution)
            return solution

    raise RuntimeError(
        "A feasible initial schedule was not found within the random sampling limit."
    )


# Sample several schedules to inspect the representation and its feasibility checks.
print("Testing random solution generation:")
for index in range(5):
    solution = generate_random_solution()
    print(
        f"Solution {index + 1}: Feasible={solution.feasible}, "
        f"Objective=${solution.objective_value:,.2f}"
    )
    print(
        "  Assignments (vessel ID, berth ID, start time in hours): "
        f"{[(vessel_id, solution.vessel_assignments[vessel_id], solution.start_times[vessel_id]) for vessel_id in sorted(solution.vessel_assignments)]}"
    )

In [ ]:
def vessel_swap_move(solution: Solution) -> Solution:
    """Swap the berth assignments of two randomly selected vessels."""
    if len(vessels) < 2:
        return solution

    vessel_id_1, vessel_id_2 = random.sample([vessel.id for vessel in vessels], 2)
    new_solution = copy.deepcopy(solution)
    new_solution.vessel_assignments[vessel_id_1], new_solution.vessel_assignments[vessel_id_2] = (
        new_solution.vessel_assignments[vessel_id_2],
        new_solution.vessel_assignments[vessel_id_1],
    )

    new_solution.feasible = is_solution_feasible(new_solution)
    new_solution.objective_value = (
        calculate_objective(new_solution) if new_solution.feasible else float("inf")
    )
    return new_solution


def time_shift_move(solution: Solution) -> Solution:
    """Move one vessel to another feasible start time on its assigned berth."""
    vessel = random.choice(vessels)
    berth_id = solution.vessel_assignments[vessel.id]
    berth = next(berth for berth in berths if berth.id == berth_id)
    feasible_times = [
        float(candidate_time)
        for candidate_time in np.arange(vessel.arrival_time, 24.0, 0.5)
        if is_feasible_assignment(vessel, berth, float(candidate_time))
    ]

    current_time = solution.start_times[vessel.id]
    alternative_times = [
        candidate_time
        for candidate_time in feasible_times
        if not math.isclose(candidate_time, current_time, rel_tol=0.0, abs_tol=1e-9)
    ]
    if not alternative_times:
        return solution

    new_solution = copy.deepcopy(solution)
    new_solution.start_times[vessel.id] = random.choice(alternative_times)
    new_solution.feasible = is_solution_feasible(new_solution)
    new_solution.objective_value = (
        calculate_objective(new_solution) if new_solution.feasible else float("inf")
    )
    return new_solution


def window_migration_move(solution: Solution) -> Solution:
    """Move one vessel's start time to a different tidal access window."""
    vessel = random.choice(vessels)
    berth_id = solution.vessel_assignments[vessel.id]
    berth = next(berth for berth in berths if berth.id == berth_id)
    current_start = solution.start_times[vessel.id]
    current_window = next(
        (
            window
            for window in tidal_windows
            if window.start_time <= current_start <= window.end_time
        ),
        None,
    )

    alternative_times = []
    for window in tidal_windows:
        if window == current_window:
            continue

        first_candidate = max(vessel.arrival_time, window.start_time)
        last_candidate = min(window.end_time, 24.0)
        for candidate_time in np.arange(first_candidate, last_candidate + 0.5, 0.5):
            candidate_time = float(candidate_time)
            if is_feasible_assignment(vessel, berth, candidate_time):
                alternative_times.append(candidate_time)

    if not alternative_times:
        return solution

    new_solution = copy.deepcopy(solution)
    new_solution.start_times[vessel.id] = random.choice(alternative_times)
    new_solution.feasible = is_solution_feasible(new_solution)
    new_solution.objective_value = (
        calculate_objective(new_solution) if new_solution.feasible else float("inf")
    )
    return new_solution


print("Neighborhood operators defined.")

In [ ]:
def hill_climbing(
    max_iterations: int = 100,
    verbose: bool = True,
) -> Tuple[Solution, List[float]]:
    """Improve one random feasible schedule using sampled neighborhood moves."""
    current_solution = generate_random_solution()
    best_solution = copy.deepcopy(current_solution)
    objective_history = [current_solution.objective_value]
    operators = [vessel_swap_move, time_shift_move, window_migration_move]

    if verbose:
        print(
            f"Initial solution: Feasible={current_solution.feasible}, "
            f"Objective=${current_solution.objective_value:,.2f}"
        )

    for iteration in range(max_iterations):
        improved = False

        # Accept the first feasible move that strictly lowers the objective.
        for operator in operators:
            neighbor = operator(current_solution)
            is_strict_improvement = (
                neighbor.feasible
                and neighbor.objective_value < current_solution.objective_value
                and not math.isclose(
                    neighbor.objective_value,
                    current_solution.objective_value,
                    rel_tol=1e-12,
                    abs_tol=1e-9,
                )
            )
            if not is_strict_improvement:
                continue

            current_solution = neighbor
            improved = True
            if current_solution.objective_value < best_solution.objective_value and not math.isclose(
                current_solution.objective_value,
                best_solution.objective_value,
                rel_tol=1e-12,
                abs_tol=1e-9,
            ):
                best_solution = copy.deepcopy(current_solution)

            if verbose:
                print(
                    f"  Iteration {iteration}: improved to "
                    f"${current_solution.objective_value:,.2f} using {operator.__name__}"
                )
            break

        objective_history.append(current_solution.objective_value)
        if not improved:
            if verbose:
                print(f"  Iteration {iteration}: no sampled improving move; terminating.")
            break

    return best_solution, objective_history


def hill_climbing_with_restarts(
    num_restarts: int = 5,
    max_iterations_per_restart: int = 100,
    verbose: bool = True,
) -> Tuple[Solution, List[List[float]]]:
    """Run independent hill climbs and return the best feasible schedule found."""
    if num_restarts < 1:
        raise ValueError("num_restarts must be at least 1.")

    global_best_solution = None
    all_histories = []

    if verbose:
        print(f"Starting hill climbing with {num_restarts} random restarts.")
        print("=" * 60)

    for restart in range(num_restarts):
        if verbose:
            print(f"\nRestart {restart + 1}/{num_restarts}:")
            print("-" * 30)

        restart_best, history = hill_climbing(
            max_iterations=max_iterations_per_restart,
            verbose=verbose,
        )
        all_histories.append(history)

        if (
            global_best_solution is None
            or restart_best.objective_value < global_best_solution.objective_value
        ):
            global_best_solution = copy.deepcopy(restart_best)

        if verbose:
            print(f"Restart {restart + 1} best: ${restart_best.objective_value:,.2f}")

    if verbose:
        print(f"\nBest objective across restarts: ${global_best_solution.objective_value:,.2f}")
        print(f"Feasible: {global_best_solution.feasible}")

    return global_best_solution, all_histories


print("Hill-climbing procedures defined.")

In [ ]:
# Run the specified five-restart experiment and measure its elapsed wall-clock time.
start_time = time.time()
best_solution, all_histories = hill_climbing_with_restarts(
    num_restarts=5,
    max_iterations_per_restart=100,
    verbose=True,
)
end_time = time.time()

print(f"\nTotal execution time: {end_time - start_time:.2f} seconds")
print(f"Number of restarts: {len(all_histories)}")

In [ ]:
def analyze_solution(solution: Solution) -> None:
    """Report each vessel's assignment, cost contribution, and berth schedule."""
    print("\nBest-solution analysis:")
    print("=" * 80)

    if not solution.feasible:
        print("The reported schedule is infeasible and cannot be analyzed as a valid plan.")
        return

    total_cost = 0.0
    for vessel_id in sorted(solution.vessel_assignments):
        vessel = next(vessel for vessel in vessels if vessel.id == vessel_id)
        berth_id = solution.vessel_assignments[vessel_id]
        berth = next(berth for berth in berths if berth.id == berth_id)
        start_time = solution.start_times[vessel_id]
        end_time = start_time + vessel.processing_time
        elapsed_time = end_time - vessel.arrival_time
        cost = vessel.demurrage_cost * elapsed_time
        total_cost += cost

        print(f"Vessel {vessel_id}:")
        print(f"  Assigned berth: {berth_id}")
        print(
            f"  Start: {start_time:.1f} h, end: {end_time:.1f} h, "
            f"elapsed time from arrival: {elapsed_time:.1f} h"
        )
        print(
            f"  Draft: {vessel.draft:.1f} m, available depth at start: "
            f"{get_tidal_depth(start_time, berth):.1f} m"
        )
        print(f"  Individual feasibility: {is_feasible_assignment(vessel, berth, start_time)}")
        print(f"  Cost contribution: ${cost:,.0f}")

    assert math.isclose(total_cost, solution.objective_value, rel_tol=1e-12, abs_tol=1e-6)
    print(f"\nTotal objective value: ${total_cost:,.0f}")

    print("\nBerth schedules and overlap check:")
    for berth in berths:
        schedule = []
        for vessel_id, berth_id in solution.vessel_assignments.items():
            if berth_id == berth.id:
                vessel = next(vessel for vessel in vessels if vessel.id == vessel_id)
                start_time = solution.start_times[vessel_id]
                schedule.append(
                    (start_time, start_time + vessel.processing_time, vessel_id)
                )
        schedule.sort()
        schedule_text = " ".join(
            f"Vessel {vessel_id} ({start:.1f}–{end:.1f} h)"
            for start, end, vessel_id in schedule
        )
        print(f"  Berth {berth.id}: {schedule_text or 'No assigned vessels'}")
        assert all(
            schedule[index][1] <= schedule[index + 1][0]
            for index in range(len(schedule) - 1)
        )


analyze_solution(best_solution)

In [ ]:
def plot_convergence(histories: List[List[float]]) -> None:
    """Plot objective histories and the best objective from each restart."""
    if not histories:
        print("No search histories are available to plot.")
        return

    finite_histories = [history for history in histories if history]
    if not finite_histories or not all(
        math.isfinite(value)
        for history in finite_histories
        for value in history
    ):
        print("Convergence plots require finite objective values in every history.")
        return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    colors = plt.cm.viridis(np.linspace(0, 1, len(histories)))

    ax1.set_title("Hill-climbing objective by restart")
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel("Objective value (cost units)")
    ax1.grid(True, alpha=0.3)

    for index, history in enumerate(histories):
        iterations = range(len(history))
        ax1.plot(
            iterations,
            history,
            color=colors[index],
            linewidth=2,
            label=f"Restart {index + 1}",
        )
        ax1.scatter([len(history) - 1], [history[-1]], color=colors[index], s=45)
    ax1.legend()

    best_values = [min(history) for history in histories]
    restart_numbers = np.arange(1, len(histories) + 1)
    bars = ax2.bar(restart_numbers, best_values, color=colors, alpha=0.75)
    ax2.set_title("Best objective value by restart")
    ax2.set_xlabel("Restart number")
    ax2.set_ylabel("Objective value (cost units)")
    ax2.set_xticks(restart_numbers)
    ax2.grid(True, axis="y", alpha=0.3)

    for bar, value in zip(bars, best_values):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"${value:,.0f}",
            ha="center",
            va="bottom",
        )

    best_restart_index = int(np.argmin(best_values))
    bars[best_restart_index].set_edgecolor("black")
    bars[best_restart_index].set_linewidth(2)
    fig.tight_layout()
    plt.show()

    print("\nConvergence summary:")
    print(
        f"Best restart: {best_restart_index + 1}, "
        f"objective=${best_values[best_restart_index]:,.0f}"
    )
    print(f"Mean best objective: ${np.mean(best_values):,.0f}")
    print(f"Standard deviation: ${np.std(best_values):,.0f}")
    spread = max(best_values) - min(best_values)
    worst_value = max(best_values)
    relative_spread = 100.0 * spread / worst_value if worst_value else 0.0
    print(
        f"Difference between worst and best restart: "
        f"${spread:,.0f} ({relative_spread:.1f}% of the worst value)"
    )


plot_convergence(all_histories)

In [ ]:
def visualize_solution(solution: Solution) -> None:
    """Draw the berth schedule and compare vessel drafts with tidal depths."""
    if not solution.feasible:
        print("A Gantt chart is available only for a feasible schedule.")
        return

    vessel_colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7"]
    latest_completion = max(
        solution.start_times[vessel.id] + vessel.processing_time
        for vessel in vessels
    )
    time_limit = max(24.0, latest_completion)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

    ax1.set_title("Hill-climbing berth allocation schedule")
    ax1.set_xlabel("Time from planning-day start (hours)")
    ax1.set_ylabel("Berth")
    ax1.set_xlim(0, time_limit)
    ax1.set_ylim(-0.5, len(berths) - 0.5)
    ax1.set_yticks([berth.id for berth in berths])
    ax1.set_yticklabels([f"Berth {berth.id}" for berth in berths])
    ax1.grid(True, alpha=0.3)

    for index, window in enumerate(tidal_windows):
        ax1.axvspan(
            window.start_time,
            window.end_time,
            alpha=0.2,
            color="lightblue",
            label="Tidal access window" if index == 0 else None,
        )

    for vessel_id in sorted(solution.vessel_assignments):
        vessel = next(vessel for vessel in vessels if vessel.id == vessel_id)
        berth_id = solution.vessel_assignments[vessel_id]
        start_time = solution.start_times[vessel_id]
        end_time = start_time + vessel.processing_time
        color = vessel_colors[vessel_id % len(vessel_colors)]
        rectangle = patches.Rectangle(
            (start_time, berth_id - 0.3),
            vessel.processing_time,
            0.6,
            linewidth=1.5,
            edgecolor="black",
            facecolor=color,
            alpha=0.8,
        )
        ax1.add_patch(rectangle)
        ax1.text(
            (start_time + end_time) / 2,
            berth_id,
            f"Vessel {vessel_id}",
            ha="center",
            va="center",
            fontsize=9,
        )

    for index, vessel in enumerate(vessels):
        ax1.axvline(
            x=vessel.arrival_time,
            color="red",
            linestyle="--",
            alpha=0.5,
            label="Vessel arrival" if index == 0 else None,
        )
    ax1.legend(loc="upper right")

    ax2.set_title("Vessel draft requirements and available berth depth")
    ax2.set_xlabel("Time from planning-day start (hours)")
    ax2.set_ylabel("Depth (meters)")
    ax2.set_xlim(0, time_limit)
    ax2.grid(True, alpha=0.3)

    time_points = np.linspace(0, time_limit, max(240, int(time_limit * 10)))
    for berth in berths:
        depths = [get_tidal_depth(point, berth) for point in time_points]
        ax2.plot(time_points, depths, linewidth=2, label=f"Berth {berth.id} available depth")

    for vessel_id in sorted(solution.vessel_assignments):
        vessel = next(vessel for vessel in vessels if vessel.id == vessel_id)
        start_time = solution.start_times[vessel_id]
        end_time = start_time + vessel.processing_time
        color = vessel_colors[vessel_id % len(vessel_colors)]
        ax2.hlines(
            vessel.draft,
            start_time,
            end_time,
            colors=color,
            linewidth=3,
            label=f"Vessel {vessel_id} draft",
        )
        ax2.axvspan(start_time, end_time, alpha=0.08, color=color)

    ax2.legend(loc="best")
    fig.tight_layout()
    plt.show()


visualize_solution(best_solution)

In [ ]:
def performance_analysis() -> None:
    """Examine restart counts, sampled operators, and approximate search complexity."""
    print("Performance analysis:")
    print("=" * 50)

    restart_counts = [1, 3, 5, 10]
    results = []
    for num_restarts in restart_counts:
        start_time = time.time()
        solution, _ = hill_climbing_with_restarts(
            num_restarts=num_restarts,
            max_iterations_per_restart=50,
            verbose=False,
        )
        elapsed_seconds = time.time() - start_time
        results.append(
            {
                "restarts": num_restarts,
                "best_value": solution.objective_value,
                "elapsed_seconds": elapsed_seconds,
                "feasible": solution.feasible,
            }
        )

    print("\nEffect of restart count:")
    print("Restarts | Best objective | Time (s) | Feasible")
    print("------------------------------------------------")
    for result in results:
        print(
            f"{result['restarts']:8d} | "
            f"${result['best_value']:13,.0f} | "
            f"{result['elapsed_seconds']:8.2f} | {result['feasible']}"
        )

    print("\nNeighborhood operator effectiveness:")
    operators = [vessel_swap_move, time_shift_move, window_migration_move]
    operator_names = ["Vessel swap", "Time shift", "Window migration"]
    base_solution = generate_random_solution()
    print(f"Base schedule objective: ${base_solution.objective_value:,.2f}")

    for operator, name in zip(operators, operator_names):
        candidate = operator(base_solution)
        if not candidate.feasible:
            print(f"{name}: sampled move produced an infeasible candidate.")
        elif candidate.objective_value < base_solution.objective_value and not math.isclose(
            candidate.objective_value,
            base_solution.objective_value,
            rel_tol=1e-12,
            abs_tol=1e-9,
        ):
            improvement = base_solution.objective_value - candidate.objective_value
            print(f"{name}: objective decreased by ${improvement:,.2f}.")
        else:
            print(f"{name}: no strict improvement in this sampled move.")

    # Each iteration samples at most one move per operator and checks full schedules.
    n_vessels = len(vessels)
    n_berths = len(berths)
    time_steps = int(24.0 / 0.5)
    print("\nApproximate time-complexity analysis:")
    print(
        f"Example size: {n_vessels} vessels, {n_berths} berths, "
        f"{time_steps} half-hour start-time steps per day."
    )
    print("A complete feasibility check sorts vessel intervals at each berth: O(N log N).")
    print("A sampled swap, time shift, or window migration then costs O(N log N) worst-case.")
    print("With R restarts and I iterations per restart, neighborhood search is O(R × I × N log N).")
    print("Initial-schedule sampling additionally scans candidate times across bounded retries.")
    print("Here R is the number of restarts, I the iteration limit, and N the number of vessels.")


performance_analysis()

## 2. Interpreting this heuristic

### What the search can and cannot establish
The result is a feasible schedule with the lowest objective value encountered during the sampled searches. A feasible schedule satisfies arrival, tidal-access, draft, and berth non-overlap conditions in this model. Hill climbing accepts only strictly improving moves, so a single run can stop at a local optimum. Random restarts explore additional starting schedules and can improve the best value found, but neither a fixed number of restarts nor a finite search proves global optimality.

The comparison with a mathematical-formulation baseline is useful for understanding solution quality when that baseline is available for the same assumptions and data. Runtime and solution quality depend on the instance, the time grid, the random starting schedules, and the available neighborhood moves; this example does not establish performance guarantees for larger instances.

### Strengths and limitations
**Strengths**
- The representation and neighborhood moves make the search logic explicit and adaptable to additional operational constraints.
- Each accepted move preserves feasibility, so the search maintains a valid schedule once it starts from a feasible one.
- Random restarts provide multiple opportunities to explore different regions of the discrete schedule space.

**Limitations**
- The method provides no optimality certificate and can miss better schedules that require a sequence of non-improving moves.
- Random sampling means results can depend on initialization and the sequence of sampled moves, even with a fixed seed.
- The 0.5-hour grid excludes start times between grid points, and the simplified depth and berth assumptions omit several operational details.
- The computational work grows with the number of vessels, the number of iterations, and the number of restarts.

### Appropriate use
This heuristic is appropriate when a feasible, good-quality schedule is useful and a formal optimality guarantee is not required. For operational use, the model assumptions, time resolution, and neighborhood design should match the planning context. The natural extensions are to incorporate additional constraints or larger problem instances while retaining the same hill-climbing and random-restart structure.